In [ ]:
src\ventas\views.py:

In [ ]:
from order_manage.models import Order  # ajusta si tu modelo es otro
from django.views.generic import TemplateView
from django.db.models.functions import TruncDay
from cart.models import Cart
from django.db.models.functions import ExtractWeekDay

# Create your views here.

class SalesChartView(TemplateView):
    template_name = "ventas/chart.html"

class SalesChartData(View):
    def get(self, request, *args, **kwargs):
        today = now()
        last_7_days = today - timedelta(days=7)

        qs = (
            Order.objects
            .filter(created_at__gte=last_7_days)
            .annotate(weekday=ExtractWeekDay("created_at"))
            .values("weekday")
            .annotate(total=Sum("total"))
        )

        # Para orden desde Lunes a Domingo
        dias = {
            1: "Domingo",
            2: "Lunes",
            3: "Martes",
            4: "Miércoles",
            5: "Jueves",
            6: "Viernes",
            7: "Sábado",
        }

        # Para que se inicialize todos los días en 0 simulando datos reales
        data_dict = {day: 0 for day in dias.values()}

        # Llenamos con datos reales
        for item in qs:
            nombre_dia = dias[item["weekday"]]
            data_dict[nombre_dia] = float(item["total"] or 0)

        # Para orden de Lunes a Domingo simulando los dias operables
        orden = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

        labels = orden
        data = [data_dict[dia] for dia in orden]

        return JsonResponse({
            "labels": labels,
            "data": data
        })


In [ ]:
src\ventas\urls.py:

In [ ]:
from django.urls import path
from . import views

from ventas.views import SalesChartData

app_name = 'ventas'

urlpatterns = [
    path('', views.lista_productos, name='productos'), # Pagina principal de los productos  
    path('agregar/<int:producto_id>/', views.agregar_al_carrito, name='agregar'), # Para agregar o productos al carrito
    path('carrito/', views.ver_carrito, name='carrito'), # Para ver los productos agregado al carrito
    path('procesar/', views.procesar_pedido, name='procesar'), # Para procesar el producto
    path('confirmacion/', views.confirmacion, name='confirmacion'), # Para simular la confirmacion 
    path('crear/', views.crear_producto, name='crear'), # para crear el producto
    path('eliminar/<int:producto_id>/', views.eliminar_producto, name='eliminar'), # Para eliminar un producto expecifico
    path('quitar/<int:producto_id>/', views.quitar_del_carrito, name='quitar_del_carrito'), # Para quitar un producto expecifico

    path("chart/", SalesChartData.as_view(), name="chart"),
    path("sales-chart/", views.SalesChartView.as_view(), name="sales-chart"),
]


In [ ]:
src\ventas\templates\ventas\chart.html:

In [ ]:
<!DOCTYPE html>
<html>
<head>
    <title>Ventas EBAC</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
</head>
<body>

<h2>Gráfico de ventas EBAC</h2>

<canvas id="salesChart"></canvas>

<script>
fetch("/ventas/chart/")
    .then(response => response.json())
    .then(data => {

        const ctx = document.getElementById('salesChart').getContext('2d');

        new Chart(ctx, {
            type: 'line',
            data: {
                labels: data.labels,
                datasets: [{
                    label: 'Ventas',
                    data: data.data,
                    borderWidth: 2
                }]
            }
        });

    })
    .catch(error => console.error("Error:", error));
</script>

</body>
</html>

